In [ ]:
import opensmile
import audiofile
import numpy as np
import pandas as pd # pandasをインポート

# --- 1. 準備 ---
VIDEO_FPS = 29.97
wav_path = "./ID36_audio.wav"

try:
    signal, sampling_rate = audiofile.read(wav_path)
except FileNotFoundError:
    print(f"エラー: ファイル '{wav_path}' が見つかりません。")
    exit()

if signal.ndim == 2:
    signal = signal[0, :]


# --- 2. スライス ---
start_video_frame = 156
end_video_frame = 222

start_sec = start_video_frame / VIDEO_FPS
end_sec = end_video_frame / VIDEO_FPS
start_sample = int(start_sec * sampling_rate)
end_sample = int(end_sec * sampling_rate)

signal_slice = signal[start_sample:end_sample]


# --- 3. 分析 & 時間情報の追加 ---
if signal_slice.size > 0:
    smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.eGeMAPSv02,
        feature_level=opensmile.FeatureLevel.LowLevelDescriptors,
    )
    result_df = smile.process_signal(
        signal_slice,
        sampling_rate
    )

    # --- ▼ ここからが追加部分 ▼ ---
    
    # 1. スライス開始時間をTimedelta型に変換
    start_timedelta = pd.to_timedelta(start_sec, unit='s')

    # 2. 元のインデックスに開始時間を加算して、新しい列を作成
    #    MultiIndexの'start'と'end'を取得して計算
    original_start = result_df.index.get_level_values('start') + start_timedelta
    original_end = result_df.index.get_level_values('end') + start_timedelta

    # 3. 新しい列をDataFrameに追加
    result_df['original_start'] = original_start
    result_df['original_end'] = original_end
    
    # --- ▲ ここまで ▲ ---

    # 'answer1'という名前の列を0番目（先頭）に挿入し、値を1に設定
    result_df.insert(0, 'label', 'answer1')
    
    print(f"✅ 元の時間情報を追加した分析結果:")
    print(result_df.head()) # 新しい列が追加されたことを確認

    # CSVファイルに保存
    output_filename = "analysis_results_with_original_time.csv"
    result_df.to_csv(output_filename)
    print(f"\n✅ 分析結果を '{output_filename}' に保存しました。")

else:
    print("⚠️ エラー: スライス後のデータが空です。")

## answer1

In [ ]:
import pandas as pd
# --- 1. 参照ファイルから'answer1'の行を検索してフレーム情報を取得 ---
try:
    # スペースやタブが複数あっても区切り文字として認識させる
    info_df = pd.read_csv("./processed_annotation/ID36_annotation_processed.csv", header=None)
    
    print(info_df)
except FileNotFoundError:
    print("エラー: 参照ファイル 'slice_info.txt' が見つかりません。")
    exit()

# 6列目(インデックス5)の値が'answer1'と完全に一致する行を抽出
target_row = info_df[info_df[5] == 'answer1']

if target_row.empty:
    print("エラー: 'slice_info.txt'内に6列目が'answer1'の行が見つかりません。")
    exit()

# 該当行の7列目(インデックス6)と8列目(インデックス7)の値を取得
start_video_frame = target_row.iloc[0, 6]
end_video_frame = target_row.iloc[0, 7]

print(start_video_frame, end_video_frame)

## answer1~10

In [ ]:
import pandas as pd

# --- 1. 参照ファイルを読み込む ---
try:
    # ヘッダーがないCSVファイルとして読み込む
    info_df = pd.read_csv("./processed_annotation/ID36_annotation_processed.csv", header=None)
except FileNotFoundError:
    print("エラー: 参照ファイルが見つかりません。")
    exit()

# --- 2. 'answer'で始まる全ての行を抽出 ---
# 6列目(インデックス5)の値を文字列に変換し、'answer'で始まる行をフィルタリング
filtered_df = info_df[info_df[5].astype(str).str.startswith('answer')]

if filtered_df.empty:
    print("エラー: 6列目が'answer'で始まる行が見つかりません。")
    exit()

print("--- 'answer'で始まる全ての区間を処理します ---")

# --- 3. 抽出した各行についてループ処理 ---
for index, row in filtered_df.iterrows():
    # 現在の行から値を取得
    label = row[5]             # 6列目: ラベル (例: 'answer1', 'answer2')
    start_video_frame = row[6] # 7列目: 開始フレーム
    end_video_frame = row[7]   # 8列目: 終了フレーム

    # 取得した値を表示（ここに個別の処理を追加できます）
    print(f"ラベル: {label}, 開始フレーム: {start_video_frame}, 終了フレーム: {end_video_frame}")

print("\n--- 全ての処理が完了しました ---")

In [ ]:
import opensmile
import audiofile
import numpy as np
import pandas as pd

# --- 1. 設定と参照ファイルの読み込み ---
VIDEO_FPS = 29.97
wav_path = "./ID38_audio.wav"
ref_file_path = "./processed_annotation/ID38_annotation_processed.csv"

try:
    # スペース区切りの参照ファイルを読み込む
    info_df = pd.read_csv(ref_file_path, header=None)
except FileNotFoundError:
    print(f"エラー: 参照ファイル '{ref_file_path}' が見つかりません。")
    exit()

# 6列目(インデックス5)の値が'answer'で始まる行をすべて抽出
answer_segments = info_df[info_df[5].astype(str).str.startswith('answer')]

if answer_segments.empty:
    print(f"エラー: '{ref_file_path}' 内に6列目が'answer'で始まる行が見つかりません。")
    exit()

# --- 2. 音声ファイルの読み込み（一度だけ）---
try:
    signal, sampling_rate = audiofile.read(wav_path)
except FileNotFoundError:
    print(f"エラー: 音声ファイル '{wav_path}' が見つかりません。")
    exit()

if signal.ndim == 2:
    signal = signal[0, :]

# --- 3. ループ処理の準備 ---
all_results = [] # 各区間の分析結果を格納する空のリストを準備
print(f"--- 音声ファイル '{wav_path}' の分析を開始します ---")

# --- 4. 抽出した各'answer'区間についてループ処理 ---
for index, segment_info in answer_segments.iterrows():
    # 現在の行から情報を取得
    label = segment_info[5]
    start_video_frame = segment_info[6]
    end_video_frame = segment_info[7]

    print(f"▶ 処理中: ラベル='{label}', フレーム={start_video_frame}-{end_video_frame}")

    # スライス処理
    start_sec = start_video_frame / VIDEO_FPS
    end_sec = end_video_frame / VIDEO_FPS
    start_sample = int(start_sec * sampling_rate)
    end_sample = int(end_sec * sampling_rate)
    signal_slice = signal[start_sample:end_sample]

    # 分析 & 結果の保存
    if signal_slice.size > 0:
        smile = opensmile.Smile(
            feature_set=r'C:\Users\robotics\proj\delirium\Lib\site-packages\opensmile\core\config\egemaps\niho\eGeMAPSv02.conf',
            feature_level=opensmile.FeatureLevel.LowLevelDescriptors,
        )
        result_df = smile.process_signal(signal_slice, sampling_rate)

        # 時間情報を追加
        start_timedelta = pd.to_timedelta(start_sec, unit='s')
        result_df['original_start'] = result_df.index.get_level_values('start') + start_timedelta
        result_df['original_end'] = result_df.index.get_level_values('end') + start_timedelta
        
        # ラベル列を先頭に追加
        result_df.insert(0, 'label', label)
        
        # ▼▼▼ 結果をリストに追加 ▼▼▼
        all_results.append(result_df)
        print(f"✅ ラベル '{label}' の処理完了。")
    else:
        print("⚠️  エラー: スライス後のデータが空です。この区間はスキップします。")

# --- 5. 全ての結果を統合して1つのファイルに保存 ---
if all_results:
    print("\n--- 全ての結果を統合します ---")
    
    # リスト内の全てのDataFrameを縦に連結
    final_df = pd.concat(all_results)
    
    # 統合した結果を1つのCSVファイルに保存
    output_filename = "ID38_all_answers.csv"
    final_df.to_csv(output_filename)
    print(f"✅ 全ての分析結果を '{output_filename}' に保存しました。")
else:
    print("\n分析できる区間がありませんでした。")

print("\n--- 全ての処理が完了しました ---")

In [ ]:
import opensmile
import audiofile
import numpy as np
import pandas as pd

# --- 1. 設定と参照ファイルの読み込み ---
VIDEO_FPS = 29.97
wav_path = "D:/Douga_Niho/voice_data"
ref_file_path = "../../processed_annotation_data/ID30_annotation_processed.csv"

try:
    # スペース区切りの参照ファイルを読み込む
    info_df = pd.read_csv(ref_file_path, header=None)
except FileNotFoundError:
    print(f"エラー: 参照ファイル '{ref_file_path}' が見つかりません。")
    exit()

# 6列目(インデックス5)の値が'answer'で始まる行をすべて抽出
answer_segments = info_df[info_df[5].astype(str).str.startswith('answer')]

if answer_segments.empty:
    print(f"エラー: '{ref_file_path}' 内に6列目が'answer'で始まる行が見つかりません。")
    exit()

# --- 2. 音声ファイルの読み込み（一度だけ）---
try:
    signal, sampling_rate = audiofile.read(wav_path)
except FileNotFoundError:
    print(f"エラー: 音声ファイル '{wav_path}' が見つかりません。")
    exit()

if signal.ndim == 2:
    signal = signal[0, :]

# --- 3. ループ処理の準備 ---
all_results = [] # 各区間の分析結果を格納する空のリストを準備
print(f"--- 音声ファイル '{wav_path}' の分析を開始します ---")

# --- 4. 抽出した各'answer'区間についてループ処理 ---
for index, segment_info in answer_segments.iterrows():
    # 現在の行から情報を取得
    label = segment_info[5]
    start_video_frame = segment_info[6]

    # ▼▼▼ ここを変更 ▼▼▼
    # 終了フレームを開始フレームに30を足した値として計算
    end_video_frame = start_video_frame + 30
    # ▲▲▲▲▲▲▲▲▲▲▲▲

    print(f"▶ 処理中: ラベル='{label}', 計算後のフレーム={start_video_frame}-{end_video_frame}")

    # スライス処理
    start_sec = start_video_frame / VIDEO_FPS
    end_sec = end_video_frame / VIDEO_FPS
    start_sample = int(start_sec * sampling_rate)
    end_sample = int(end_sec * sampling_rate)
    signal_slice = signal[start_sample:end_sample]

    # 分析 & 結果の保存
    if signal_slice.size > 0:
        smile = opensmile.Smile(
            feature_set=opensmile.FeatureSet.ComParE_2016,
            feature_level=opensmile.FeatureLevel.LowLevelDescriptors,
        )
        result_df = smile.process_signal(signal_slice, sampling_rate)

        # 時間情報を追加
        start_timedelta = pd.to_timedelta(start_sec, unit='s')
        result_df['original_start'] = result_df.index.get_level_values('start') + start_timedelta
        result_df['original_end'] = result_df.index.get_level_values('end') + start_timedelta
        
        # ラベル列を先頭に追加
        result_df.insert(0, 'label', label)
        
        # 結果をリストに追加
        all_results.append(result_df)
        print(f"✅ ラベル '{label}' の処理完了。")
    else:
        print("⚠️  エラー: スライス後のデータが空です。この区間はスキップします。")

# --- 5. 全ての結果を統合して1つのファイルに保存 ---
if all_results:
    print("\n--- 全ての結果を統合します ---")
    
    # リスト内の全てのDataFrameを縦に連結
    final_df = pd.concat(all_results)
    
    # 統合した結果を1つのCSVファイルに保存
    output_filename = "ID30_comapare_answers.csv"
    final_df.to_csv(output_filename)
    print(f"✅ 全ての分析結果を '{output_filename}' に保存しました。")
else:
    print("\n分析できる区間がありませんでした。")

print("\n--- 全ての処理が完了しました ---")

In [ ]:
import opensmile
import audiofile
import numpy as np
import pandas as pd
import os
import glob

# --- 1. フォルダ設定 ---
AUDIO_INPUT_FOLDER = "D:/Douga_Niho/voice_data"  # 音声ファイルが入ったフォルダ
ANNOTATION_INPUT_FOLDER = "../../processed_annotation_data"  # 注釈ファイルが入ったフォルダ
OUTPUT_FOLDER = "../../voice_csv_Comapre"  # 分析結果を保存するフォルダ

VIDEO_FPS = 29.97

def process_single_pair(wav_path, ref_file_path, output_csv_path):
    """
    1組の音声ファイルと注釈ファイルを処理し、結果をCSVとして保存する関数
    """
    # --- 参照ファイルの読み込み ---
    try:
        info_df = pd.read_csv(ref_file_path, header=None)
    except FileNotFoundError:
        print(f"⚠️ エラー: 参照ファイル '{ref_file_path}' が見つかりません。")
        return

    answer_segments = info_df[info_df[5].astype(str).str.startswith('answer')]
    if answer_segments.empty:
        print(f"⚠️ '{os.path.basename(ref_file_path)}' 内に 'answer' で始まる行が見つかりません。")
        return

    # --- 音声ファイルの読み込み ---
    try:
        signal, sampling_rate = audiofile.read(wav_path)
    except FileNotFoundError:
        print(f"⚠️ エラー: 音声ファイル '{wav_path}' が見つかりません。")
        return

    if signal.ndim == 2:
        signal = signal[0, :]

    # --- 各'answer'区間の処理 ---
    all_results = []
    for index, segment_info in answer_segments.iterrows():
        label = segment_info[5]
        start_video_frame = segment_info[6]
        end_video_frame = start_video_frame + 30

        print(f"  ▶ 処理中: ラベル='{label}', 計算後のフレーム={start_video_frame}-{end_video_frame}")

        start_sec = start_video_frame / VIDEO_FPS
        end_sec = end_video_frame / VIDEO_FPS
        start_sample = int(start_sec * sampling_rate)
        end_sample = int(end_sec * sampling_rate)
        signal_slice = signal[start_sample:end_sample] 
        
        # opensmile
        if signal_slice.size > 0:
            smile = opensmile.Smile(
                feature_set=opensmile.FeatureSet.ComParE_2016,
                feature_level=opensmile.FeatureLevel.LowLevelDescriptors,
            )
            result_df = smile.process_signal(signal_slice, sampling_rate)
            start_timedelta = pd.to_timedelta(start_sec, unit='s')
            result_df['original_start'] = result_df.index.get_level_values('start') + start_timedelta
            result_df['original_end'] = result_df.index.get_level_values('end') + start_timedelta
            result_df.insert(0, 'label', label)
            all_results.append(result_df)
        else:
            print(f"  ⚠️  エラー: ラベル '{label}' のスライス後データが空です。スキップします。")

    # --- 結果の統合と保存 ---
    if all_results:
        final_df = pd.concat(all_results)
        final_df.to_csv(output_csv_path)
        print(f"✅ 結果を '{output_csv_path}' に保存しました。")
    else:
        print(f"  - 分析できる区間がありませんでした。")


# --- メイン処理 ---
if __name__ == "__main__":
    print("--- 全てのファイルの処理を開始します ---")

    # 結果保存用フォルダがなければ作成
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    # 音声フォルダ内の全ての.wavファイルを検索
    audio_files = glob.glob(os.path.join(AUDIO_INPUT_FOLDER, '*.wav'))

    if not audio_files:
        print(f"エラー: 音声フォルダ '{AUDIO_INPUT_FOLDER}' に .wav ファイルが見つかりません。")
    else:
        # 見つかった音声ファイルごとにループ処理
        for audio_path in audio_files:
            base_name = os.path.basename(audio_path)
            
            # ファイル名からIDを抽出 (例: "ID70_audio.wav" -> "ID70")
            try:
                file_id = base_name.split('_')[0]
            except IndexError:
                print(f"⚠️ ファイル名 '{base_name}' の形式が不正です。スキップします。")
                continue

            print(f"\n--- ID: {file_id} の処理を開始 ---")

            # 対応する注釈ファイルと出力ファイルのパスを生成
            annotation_path = os.path.join(ANNOTATION_INPUT_FOLDER, f"{file_id}_annotation_processed.csv")
            output_path = os.path.join(OUTPUT_FOLDER, f"{file_id}_Compare_answers.csv")

            # 注釈ファイルが存在するか確認
            if os.path.exists(annotation_path):
                # 存在すれば、処理関数を呼び出す
                process_single_pair(audio_path, annotation_path, output_path)
            else:
                print(f"⚠️ 注釈ファイル '{annotation_path}' が見つかりません。スキップします。")

    print("\n--- 全ての処理が完了しました ---")

## ラベルごとの個数

In [ ]:


import pandas as pd

def count_specific_labels_in_column(file_path, column_name='label'):
    """
    CSVファイルを読み込み、指定された単一の列（デフォルトは'label'）内で
    'answer1'から'answer10'までの各ラベルの個数をカウントします。

    Args:
        file_path (str): 読み込むCSVファイルのパス。
        column_name (str): 集計対象の列名。
    """
    try:
        # CSVファイルをpandasのDataFrameとして読み込む
        # 文字コードが原因でエラーが出る場合は、 'utf-8' を 'shift_jis' などに変更してください
        df = pd.read_csv(file_path, encoding='utf-8')

        print(f"ファイル '{file_path}' の '{column_name}' 列を集計します。\n")

        # 指定された列がDataFrameに存在するかどうかを確認
        if column_name in df.columns:
            # 対象列の全ラベルの個数を一度に集計
            all_label_counts = df[column_name].value_counts()

            # 集計したいラベルのリストを作成
            target_labels = [f'answer{i}' for i in range(1, 11)]
            
            print("--- 集計結果 ---")
            # 結果を格納するための辞書
            results = {}
            
            # 集計したいラベルがそれぞれ何個あるかを確認
            for label in target_labels:
                # all_label_countsにラベルが存在すればその個数を、存在しなければ0を取得
                count = all_label_counts.get(label, 0)
                results[label] = count
            
            # 結果を整形して表示
            for label, count in results.items():
                print(f"{label}: {count}個")
                
        else:
            print(f"エラー: 列 '{column_name}' はファイルに存在しませんでした。")
            print(f"利用可能な列: {df.columns.tolist()}")

    except FileNotFoundError:
        print(f"エラー: ファイル '{file_path}' が見つかりませんでした。パスを確認してください。")
    except Exception as e:
        print(f"エラーが発生しました: {e}")

# --- ここから実行 ---

# ↓↓↓ あなたが集計したいCSVファイルへのパスをここに指定してください ↓↓↓
csv_file_to_analyze = 'voice_csv/ID91_30_answers.csv' 

# 関数を呼び出して集計を実行
# もし集計したい列名が'label'でない場合は、第2引数で指定できます。
# 例: count_specific_labels_in_column(csv_file_to_analyze, column_name='your_column_name')
count_specific_labels_in_column(csv_file_to_analyze)

In [1]:
import pandas as pd
import glob
import os

def load_filter_and_drop_csvs(folder_path, column_to_filter='label', columns_to_drop=None):
    """
    指定されたフォルダ内の全CSVを結合し、特定のラベルでフィルタリングし、
    不要な列を削除します。

    Args:
        folder_path (str): CSVファイルが格納されているフォルダのパス。
        column_to_filter (str): フィルタリングの基準となる列名。
        columns_to_drop (list, optional): 削除したい列名のリスト。

    Returns:
        pandas.DataFrame: 最終的に処理されたDataFrame。
    """
    # 1. フォルダ内の全CSVファイルを読み込み、結合する
    # --------------------------------------------------
    if not os.path.isdir(folder_path):
        print(f"エラー: フォルダ '{folder_path}' が見つかりません。パスを確認してください。")
        return None

    csv_pattern = os.path.join(folder_path, '*.csv')
    csv_files = glob.glob(csv_pattern)

    if not csv_files:
        print(f"フォルダ '{folder_path}' にCSVファイルが見つかりませんでした。")
        return None

    print(f"フォルダ '{folder_path}' から以下のファイルを読み込みます：")
    for f in csv_files:
        print(f" - {os.path.basename(f)}")
    print("-" * 30 + "\n")

    df_list = []
    for file in csv_files:
        try:
            df = pd.read_csv(file, encoding='utf-8')
            df['source_file'] = os.path.basename(file)
            df_list.append(df)
        except Exception as e:
            print(f"ファイル '{file}' の読み込み中にエラーが発生しました: {e}")
            continue

    if not df_list:
        print("読み込み可能なCSVファイルがありませんでした。")
        return None

    combined_df = pd.concat(df_list, ignore_index=True)
    print(f"全ファイルの結合が完了しました。結合後の合計行数: {len(combined_df)}")

    # 2. 'label'列の値に基づいて行をフィルタリングする
    # --------------------------------------------------
    if column_to_filter not in combined_df.columns:
        print(f"エラー: フィルタリング対象の列 '{column_to_filter}' がデータに存在しません。")
        return combined_df

    target_labels = [f'answer{i}' for i in range(1, 11)]
    print(f"\n'{column_to_filter}' 列の値が {target_labels} の行のみを抽出します...")
    
    filtered_df = combined_df[combined_df[column_to_filter].isin(target_labels)].copy()
    print(f"フィルタリング後の行数: {len(filtered_df)}")

    # 3. 不要な列を削除する
    # --------------------------------------------------
    if columns_to_drop:
        existing_cols_to_drop = [col for col in columns_to_drop if col in filtered_df.columns]
        
        if existing_cols_to_drop:
            final_df = filtered_df.drop(columns=existing_cols_to_drop)
            print(f"\n以下の列を削除しました: {existing_cols_to_drop}")
        else:
            final_df = filtered_df
            print("\n指定された削除対象の列はデータフレームに存在しませんでした。")
    else:
        final_df = filtered_df
        print("\n削除対象の列は指定されていません。")
    
    # 最終的な行番号をリセット
    final_df.reset_index(drop=True, inplace=True)
    
    return final_df

# --- ここから実行 ---

# 1. CSVファイルが保存されているフォルダのパスを指定
target_folder_path = './voice_csv'

# 2. 削除したい列の名前をリストで指定
columns_to_remove = ['start', 'end', 'original_start', 'original_end'] 

# 関数を呼び出して全処理を実行
final_data_df = load_filter_and_drop_csvs(
    target_folder_path, 
    column_to_filter='label', 
    columns_to_drop=columns_to_remove
)

# 処理後のデータの情報を表示
if final_data_df is not None:
    print("\n--- 最終的なDataFrameの情報 ---")
    print(f"最終的な合計行数: {len(final_data_df)}")
    print(f"最終的な合計列数: {len(final_data_df.columns)}")
    print("\n最初の5行:")
    pd.set_option('display.max_rows', None)
    print(final_data_df)
    
    if 'label' in final_data_df.columns:
        print("\n'label'列の値の内訳:")
        print(final_data_df['label'].value_counts())

エラー: フォルダ './voice_csv' が見つかりません。パスを確認してください。


In [ ]:
import pandas as pd
import glob
import os
import re

def process_and_save_csvs(folder_path, output_filepath, column_to_filter='label', columns_to_drop=None):
    """
    指定フォルダ内のCSVを結合、フィルタリング、列削除し、新しいCSVファイルに保存します。
    """
    # 1. フォルダ内の全CSVファイルを読み込み、結合する
    # --------------------------------------------------
    if not os.path.isdir(folder_path):
        print(f"エラー: フォルダ '{folder_path}' が見つかりません。パスを確認してください。")
        return None

    csv_pattern = os.path.join(folder_path, '*.csv')
    csv_files = glob.glob(csv_pattern)
    
    def extract_number(filename):
        match = re.search(r'\d+', os.path.basename(filename))
        return int(match.group()) if match else 0
    csv_files.sort(key=extract_number)

    if not csv_files:
        print(f"フォルダ '{folder_path}' にCSVファイルが見つかりませんでした。")
        return None

    print(f"フォルダ '{folder_path}' から以下のファイルを読み込みます：")
    for f in csv_files:
        print(f" - {os.path.basename(f)}")
    print("-" * 30 + "\n")

    df_list = []
    for file in csv_files:
        try:
            df = pd.read_csv(file, encoding='utf-8')
            df['source_file'] = os.path.basename(file)
            df_list.append(df)
        except Exception as e:
            print(f"ファイル '{file}' の読み込み中にエラーが発生しました: {e}")
            continue

    if not df_list:
        print("読み込み可能なCSVファイルがありませんでした。")
        return None

    combined_df = pd.concat(df_list, ignore_index=True)
    print(f"全ファイルの結合が完了しました。結合後の合計行数: {len(combined_df)}")

    # 2. 'label'列の値に基づいて行をフィルタリングする
    # --------------------------------------------------
    if column_to_filter not in combined_df.columns:
        print(f"エラー: フィルタリング対象の列 '{column_to_filter}' がデータに存在しません。")
        return combined_df

    target_labels = [f'answer{i}' for i in range(1, 11)]
    print(f"\n'{column_to_filter}' 列の値が {target_labels} の行のみを抽出します...")
    
    # ▼▼▼ ここを変更 ▼▼▼
    # .astype(str)で列を文字列型に変換し、.str.strip()で前後の空白を削除してからフィルタリング
    filtered_df = combined_df[combined_df[column_to_filter].astype(str).str.strip().isin(target_labels)].copy()
    # ▲▲▲▲▲▲▲▲▲▲▲▲
    
    print(f"フィルタリング後の行数: {len(filtered_df)}")

    # 3. 不要な列を削除する
    # --------------------------------------------------
    if columns_to_drop:
        existing_cols_to_drop = [col for col in columns_to_drop if col in filtered_df.columns]
        if existing_cols_to_drop:
            final_df = filtered_df.drop(columns=existing_cols_to_drop)
            print(f"\n以下の列を削除しました: {existing_cols_to_drop}")
        else:
            final_df = filtered_df
            print("\n指定された削除対象の列はデータフレームに存在しませんでした。")
    else:
        final_df = filtered_df
        print("\n削除対象の列は指定されていません。")
    
    final_df.reset_index(drop=True, inplace=True)
    
    # 4. 新しいCSVファイルに保存する
    # --------------------------------------------------
    try:
        final_df.to_csv(output_filepath, index=False, encoding='utf-8-sig')
        print(f"\n処理後のデータを '{output_filepath}' に保存しました。")
    except Exception as e:
        print(f"\nファイル保存中にエラーが発生しました: {e}")

    return final_df

# --- ここから実行 ---

# 1. 読み込むCSVファイルが保存されているフォルダのパス
target_folder_path = './voice_csv'

# 2. 削除したい列の名前のリスト
columns_to_remove = ['start', 'end', 'original_start', 'original_end'] 

# 3. 保存する新しいファイルの名前（パス）
output_file_name = 'processed_voice_data.csv'

# 関数を呼び出して全処理を実行
final_data_df = process_and_save_csvs(
    target_folder_path,
    output_file_name,
    column_to_filter='label', 
    columns_to_drop=columns_to_remove
)

# 処理後のデータの情報を表示
if final_data_df is not None:
    print("\n--- 最終的なDataFrameの情報 ---")
    print(f"最終的な合計行数: {len(final_data_df)}")
    print(f"最終的な合計列数: {len(final_data_df.columns)}")
    print("\n最初の5行:")
    print(final_data_df.head())